In [4]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [5]:
n_samples = 1200
classes   = ['Plastic Bottle', 'Plastic Bag', 'Fishing Net', 'Foam/Styrofoam', 'Clean Water']

def make_features(label_idx, n):
    base = np.array([
        [0.8, 0.2, 0.6, 0.3, 0.7],   # Plastic Bottle
        [0.3, 0.8, 0.2, 0.5, 0.6],   # Plastic Bag
        [0.5, 0.4, 0.9, 0.2, 0.3],   # Fishing Net
        [0.7, 0.3, 0.3, 0.9, 0.5],   # Foam/Styrofoam
        [0.1, 0.1, 0.1, 0.1, 0.9],   # Clean Water
    ])
    noise = np.random.normal(0, 0.08, (n, 5))
    return np.clip(base[label_idx] + noise, 0, 1)

X_list, y_list = [], []
for i in range(len(classes)):
    n = n_samples // len(classes)
    X_list.append(make_features(i, n))
    y_list.extend([i] * n)

X = np.vstack(X_list)
y = np.array(y_list)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset : {n_samples} samples, {len(classes)} classes")
print(f"Train   : {len(X_train)}  |  Test: {len(X_test)}")

Dataset : 1200 samples, 5 classes
Train   : 960  |  Test: 240


In [6]:
def relu(z):     return np.maximum(0, z)
def relu_d(z):   return (z > 0).astype(float)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
def one_hot(y, n):
    m = np.zeros((len(y), n)); m[np.arange(len(y)), y] = 1; return m

class DeepNeuralNetwork:
    """
    Fully-connected deep neural network trained via back-propagation.
    """
    def __init__(self, layers, lr=0.01, epochs=120, batch=64):
        self.layers = layers
        self.lr     = lr
        self.epochs = epochs
        self.batch  = batch
        self.weights, self.biases = [], []
        self.train_loss, self.val_loss = [], []
        self.train_acc,  self.val_acc  = [], []
        # He initialisation
        for i in range(len(layers) - 1):
            w = np.random.randn(layers[i], layers[i+1]) * np.sqrt(2 / layers[i])
            b = np.zeros((1, layers[i+1]))
            self.weights.append(w); self.biases.append(b)

    def forward(self, X):
        self.Z, self.A = [], [X]
        for i in range(len(self.weights) - 1):
            z = self.A[-1] @ self.weights[i] + self.biases[i]
            self.Z.append(z); self.A.append(relu(z))
        z = self.A[-1] @ self.weights[-1] + self.biases[-1]
        self.Z.append(z); self.A.append(softmax(z))
        return self.A[-1]

    def loss(self, y_oh, out):
        return -np.mean(np.sum(y_oh * np.log(out + 1e-9), axis=1))

    def backward(self, y_oh):
        m   = y_oh.shape[0]
        dA  = self.A[-1] - y_oh
        for i in reversed(range(len(self.weights))):
            dW = self.A[i].T @ dA / m
            db = dA.mean(axis=0, keepdims=True)
            self.weights[i] -= self.lr * dW
            self.biases[i]  -= self.lr * db
            if i > 0:
                dA = (dA @ self.weights[i].T) * relu_d(self.Z[i-1])

    def fit(self, Xtr, ytr, Xv, yv):
        n_cls  = len(np.unique(ytr))
        ytr_oh = one_hot(ytr, n_cls)
        yv_oh  = one_hot(yv,  n_cls)
        for ep in range(self.epochs):
            idx = np.random.permutation(len(Xtr))
            for s in range(0, len(Xtr), self.batch):
                bi  = idx[s:s + self.batch]
                out = self.forward(Xtr[bi])
                self.backward(ytr_oh[bi])
            tr_out = self.forward(Xtr); vl_out = self.forward(Xv)
            self.train_loss.append(self.loss(ytr_oh, tr_out))
            self.val_loss.append(self.loss(yv_oh, vl_out))
            self.train_acc.append((tr_out.argmax(1) == ytr).mean() * 100)
            self.val_acc.append((vl_out.argmax(1)   == yv ).mean() * 100)
            if (ep + 1) % 20 == 0:
                print(f"Epoch {ep+1:3d}/{self.epochs}  "
                      f"Loss: {self.train_loss[-1]:.4f}  "
                      f"Val Acc: {self.val_acc[-1]:.2f}%")

    def predict(self, X):
        return self.forward(X).argmax(axis=1)

# Train
model = DeepNeuralNetwork([5, 128, 64, 32, 5], lr=0.015, epochs=120, batch=64)
model.fit(X_train, y_train, X_test, y_test)

# Evaluate
y_pred   = model.predict(X_test)
accuracy = (y_pred == y_test).mean() * 100
cm       = confusion_matrix(y_test, y_pred)
report   = classification_report(y_test, y_pred, target_names=classes, output_dict=True)

print(f"\nFinal Test Accuracy : {accuracy:.2f}%")
print(classification_report(y_test, y_pred, target_names=classes))


Epoch  20/120  Loss: 0.0554  Val Acc: 100.00%
Epoch  40/120  Loss: 0.0188  Val Acc: 100.00%
Epoch  60/120  Loss: 0.0107  Val Acc: 100.00%
Epoch  80/120  Loss: 0.0074  Val Acc: 100.00%
Epoch 100/120  Loss: 0.0056  Val Acc: 100.00%
Epoch 120/120  Loss: 0.0044  Val Acc: 100.00%

Final Test Accuracy : 100.00%
                precision    recall  f1-score   support

Plastic Bottle       1.00      1.00      1.00        48
   Plastic Bag       1.00      1.00      1.00        48
   Fishing Net       1.00      1.00      1.00        48
Foam/Styrofoam       1.00      1.00      1.00        48
   Clean Water       1.00      1.00      1.00        48

      accuracy                           1.00       240
     macro avg       1.00      1.00      1.00       240
  weighted avg       1.00      1.00      1.00       240



In [7]:
OCEAN='#0a1628'; TEAL='#00d4aa'; CYAN='#00b4d8'; AMBER='#f59e0b'
CORAL='#f87171'; WHITE='#f0f9ff'; CARD='#0f2744'; MID='#162f52'

fig = plt.figure(figsize=(22, 26), facecolor=OCEAN)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.42, wspace=0.35,
                        left=0.06, right=0.97, top=0.93, bottom=0.04)

ax_head = fig.add_axes([0, 0.945, 1, 0.055])
ax_head.set_facecolor(CARD); ax_head.axis('off')
ax_head.text(0.5, 0.62, '🌊  Marine Plastic Waste Detection — Deep Learning',
             ha='center', va='center', fontsize=19, fontweight='bold', color=WHITE,
             fontfamily='monospace')
ax_head.text(0.5, 0.18,
             f'Dataset: {n_samples} samples  |  Architecture: 5→128→64→32→5  |  Test Accuracy: {accuracy:.2f}%',
             ha='center', va='center', fontsize=11, color=TEAL, fontfamily='monospace')

# … (plotting code identical to inline version) …
plt.savefig('marine_plastic_detection.png', dpi=150, bbox_inches='tight', facecolor=OCEAN)
print("Figure saved → marine_plastic_detection.png")

Figure saved → marine_plastic_detection.png
